# Различные полезности при обучении НС средствами <code>Keras</code>

# Загрузка и подготовка данных

## Получение данных

Описание датасета:

- CRIM     per capita crime rate by town
- ZN       proportion of residential land zoned for lots over 25,000 sq.ft.
- INDUS    proportion of non-retail business acres per town
- CHAS     Charles River dummy variable (= 1 if tract bounds river; 0 otherwise)
- NOX      nitric oxides concentration (parts per 10 million)
- RM       average number of rooms per dwelling
- AGE      proportion of owner-occupied units built prior to 1940
- DIS      weighted distances to five Boston employment centres
- RAD      index of accessibility to radial highways
- TAX      full-value property-tax rate per 10,000
- PTRATIO  pupil-teacher ratio by town
- B        1000(Bk - 0.63)^2 where Bk is the proportion of blacks by town
- LSTAT    % lower status of the population
- MEDV     Median value of owner-occupied homes in $1000's

In [ ]:
from keras.datasets import boston_housing

(X_train, y_train), (X_test, y_test) = boston_housing.load_data()

X_train.shape, X_test.shape

In [ ]:
X_train[:1]

## Масштабирование данных

In [ ]:
mean = X_train.mean(axis=0)
std = X_train.std(axis=0)

mean, std

In [ ]:
X_train -= mean
X_train /= std

X_test -= mean
X_test /= std

In [ ]:
X_train.mean(axis=0)

In [ ]:
X_train.std(axis=0)

# Архитектура сети


Определение сети через класс Sequential и добавление слоев в него через add

In [ ]:
from keras.models import Sequential
from keras.layers import Dense
import tensorflow as tf
tf.random.set_seed(9)


model = Sequential()
model.add(Dense(32, activation='relu', input_shape=(X_train.shape[1], )))
model.add(Dense(16, activation='relu'))
model.add(Dense(1))

model.summary()

In [ ]:
model.compile(optimizer='adam', loss='mse', metrics=['mae'])

In [ ]:
404 / 32

### Обучение сети

In [ ]:
%%time

num_epochs = 10

model.fit(X_train, y_train,
          epochs=num_epochs)

## Некоторые детали метода fit

Сейчас прошло 13 батчей через сеть

#### **batch_size**
> batch_size: Integer или None.<br>
Количество сэмплов за один шаг градиентного спуска<br>
Если None, то batch_size примет дефолтное значение 32<br>

In [ ]:
X_train.shape

In [ ]:
404 / 32

<img src='https://drive.google.com/uc?export=view&id=1j8SxKYEi12jzJXi_bPO28q5SV9emuu0Y'>

In [ ]:
%%time
num_epochs = 10

model.fit(X_train, y_train,
          epochs=num_epochs,
          batch_size=64)

In [ ]:
404 / 64

In [ ]:
%%time
num_epochs = 10

model.fit(X_train, y_train,
          epochs=num_epochs,
          batch_size=404)

#### **steps_per_epoch**

>  steps_per_epoch: Integer or `None`.<br>
        Общее количество шагов (батчей сэмплов) на одной эпохе. <br>
        Если стоит None, то steps_per_epoch равняется количеству сэмплов в датасете деленное на batch size

In [ ]:
404 / 10

In [ ]:
%%time
num_epochs = 10

model.fit(X_train, y_train,
          epochs=num_epochs,
          steps_per_epoch=10)

#### **validation_split**

> validation_split: Float между 0 и 1.<br>
        Доля обучающих данных, которая будет использоваться как валидационная часть<br>
        Данные для валидации берутся из данных с конца в `x` и `y` **до перемешивания**.

In [ ]:
404 - 81

In [ ]:
%%time
num_epochs = 10

model.fit(X_train, y_train,
          epochs=num_epochs,
          batch_size=1,
          validation_split=0.2)

#### **validation_data**

> validation_data: Данные, на которых считается функция потерь и метрики в конце каждой эпохи<br>
Модель не будет обучаться на этих данных


In [ ]:
%%time
num_epochs = 10

model.fit(X_train, y_train,
          epochs=num_epochs,
          batch_size=1,
          validation_data=(X_test, y_test))

#### **validation_freq**
>validation_freq: Используется только если предоставлены validation data: `Integer`
        или `collections.abc.Container` (например, list, tuple, и др.).<br>
        `validation_freq=2` будет запускать валидацию каждые 2 эпохи.<br>
        `validation_freq=[1, 2, 10]` запускает валидацию на 1, 2 и 10 эпохах.

In [ ]:
%%time
num_epochs = 10

model.fit(X_train, y_train,
          epochs=num_epochs,
          batch_size=1,
          validation_data=(X_test, y_test),
          validation_batch_size=X_test.shape[0],
          validation_freq=2)

In [ ]:
%%time
num_epochs = 10

model.fit(X_train, y_train,
          epochs=num_epochs,
          batch_size=1,
          validation_data=(X_test, y_test),
          validation_freq=[1, 5, 10])

## Обучение нейросети на генераторах

Про Sequence больше [тут](https://www.tensorflow.org/api_docs/python/tf/keras/utils/Sequence)

In [ ]:
import numpy as np
import keras
from tensorflow.keras.utils import Sequence



class DataGenerator(Sequence):
    def __init__(self, data, labels, batch_size=32):
        super().__init__(**kwargs)  # Вызов конструктора базового класса
        self.batch_size = batch_size
        self.data = data
        self.labels = labels

    def __len__(self):
        return int(np.round(len(self.data) / self.batch_size))

    def __getitem__(self, index):
        X = self.data[index * self.batch_size : (index+1) * self.batch_size]
        y = self.labels[index * self.batch_size : (index+1) * self.batch_size]

        return X, y

In [ ]:
train_datagen = DataGenerator(X_train, y_train)
test_datagen = DataGenerator(X_test, y_test)

In [ ]:
len(train_datagen)

In [ ]:
len(test_datagen)

In [ ]:
for X, y in train_datagen:
    print(X.shape)
    print(y.shape)
    break

In [ ]:
404 / 32

In [ ]:
%%time
num_epochs = 10

model.fit(train_datagen,
          epochs=num_epochs,
          validation_data=test_datagen)

#Callbacks

**Callbacks**(Обратные вызовы) используются для отслеживания процесса обучения и выполнения каких-либо манипуляций в процессе обучения не прерывая вручную сам процесс (логгирование, сохранение моделей и тп)

Документация [здесь](https://keras.io/api/callbacks/)

## History

In [ ]:
%%time
model = Sequential()
model.add(Dense(32, activation='relu', input_shape=(X_train.shape[1], )))
model.add(Dense(16, activation='relu'))
model.add(Dense(1))
model.compile(optimizer='adam', loss='mse', metrics=['mae'])

num_epochs = 20

history = model.fit(train_datagen,
                    epochs=num_epochs,
                    validation_data=test_datagen)

In [ ]:
history

In [ ]:
history.history.keys()

In [ ]:
import matplotlib.pyplot as plt

plt.plot(history.history['loss'], label='train loss')
plt.plot(history.history['val_loss'], label='test loss')
plt.legend();

In [ ]:
plt.plot(history.history['mae'], label='train mae')
plt.plot(history.history['val_mae'], label='test mae')
plt.legend();

## ModelCheckpoint

Сохранение модели после каждой эпохи.

`Filepath` может содержать именованные опции форматирования, заполняемые значениями epoch и ключами в `logs` (передаваемыми `on_epoch_end`).

К примеру: если `filepath` назван `weights.{epoch:02d}-{val_loss:.2f}.hdf5`, тогда модель будет сохраняться с номером эпохи и `validation_loss` в имени файла.

**Аргументы:**

— `filepath`: строка, путь сохранения модели

— `monitor`: параметр для мониторинга

— `verbose`: режим отображения, 0 или 1

— `save_best_only`: если `save_best_only=True`, если результат текущей эпохи хуже предыдущей, он не будет сохранен.

— `save_weights_only`: если `True`, тогда будут сохраняться только веса модели (`model.save_weights(filepath)`), в противном случае будет сохраняться вся модель (`model.save(filepath)`).

— `mode`: один из `{auto, min, max}`. Если `save_best_only=True`, решение о перезаписи текущего файла будет приниматься в зависимости от уменьшения/увеличения параметра мониторинга. Для `val_acc`, необходим `max`, для `val_loss` необходим `min`. В `auto` режиме, `mode` выбирается в зависимости от имени `monitor`.

— `save_freq`: `epoch` или integer. Если `epoch`, то сохраняется модель после каждой **эпохи**. Когда integer, то сохранение модели через это кол-во батчей.


In [ ]:
from keras import callbacks

model_checkpoint = callbacks.ModelCheckpoint(filepath='model_best_{epoch}.keras',
                                             monitor='val_mae',
                                             verbose=1,
                                             save_best_only=True,
                                             save_weights_only=False,
                                             mode='auto',
                                             save_freq='epoch'
                                             )

In [ ]:
model.compile(optimizer='adam', loss='mse', metrics=['mae'])

model.fit(X_train, y_train,
          epochs=5,
          validation_data=(X_test, y_test),
          callbacks=[model_checkpoint])

## Чтение модели из файла

Сравним предсказания исходной модели и загруженной из файла

In [ ]:
model.predict(X_test)[:5]

In [ ]:
from keras.models import load_model
model_loaded = load_model('/content/model_best_5.keras')

model_loaded.predict(X_test)[:5]

## EarlyStopping


Прекращение обучения, когда параметр `monitor` перестает улучшаться.

**Аргументы**

— `monitor`: параметр для мониторинга

— `min_delta`: минимальное значение изменения величины `monitor`, расцениваемое как улучшение, то есть, если абсолютное изменение меньше `min_delta`, то улучшение не засчитывается

— `patience`: число эпох, за которые величина `monitor` не улучшается, после которых обучение будет остановлено. Проверочные величины могут производиться не после каждой эпохи если `validation_freq` (`model.fit(validation_freq=5`)) больше единицы.

— `verbose`: режим отображения, 0 или 1.

— `mode`: один из `{auto, min, max}`. В режиме `min`, обучение остановится когда величина `monitor` перестанет уменьшаться; в режиме `max`, обучение остановится когда величина `monitor` перестанет увеличиваться; в режиме `auto`, `mode` выбирается в зависимости от имени `monitor`.

— `baseline`: значение, которое должна достичь величина `monitor`. Обучение прекратится, если модель не достигла baseline.

— `restore_best_weights`: восстанавливать ли веса модели с эпохи с лучшем значением параметра `monitor`. Если False, веса модели будут загружены из последней шага обучения.

In [ ]:
early_stop = callbacks.EarlyStopping(monitor='val_mae',
                                     min_delta=0.1,
                                     patience=2,
                                     verbose=1,
                                     mode='auto',
                                     restore_best_weights=True)


model.fit(X_train, y_train,
          epochs=10,
          validation_data=(X_test, y_test),
          callbacks=[early_stop])

## ReduceLROnPlateau

Уменьшение скорости обучения, когда метрика перестала улучшаться.

**Аргументы**

— `factor`: коэффициент уменьшения скорости обучения.

— `cooldown`: число эпох после уменьшения скорости обучения, которые должны пройти, прежде чем стандартный процесс уменьшения возобновится.

— `min_lr`: нижняя граница скорости обучения

In [ ]:
model.compile(optimizer='adam', loss='mse', metrics=['mae'])

reduce_lr = callbacks.ReduceLROnPlateau(monitor='val_mae',
                                        factor=0.1,
                                        patience=0,
                                        verbose=1,
                                        mode='auto',
                                        min_delta=0.1,
                                        cooldown=2,
                                        min_lr=1e-10)

model.fit(X_train, y_train,
          epochs=10,
          validation_data=(X_test, y_test),
          callbacks=[reduce_lr])

## Кастомные коллбэки

Читать вот [тут](https://www.tensorflow.org/guide/keras/writing_your_own_callbacks)